In [1]:
import re
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
import os
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = False

In [2]:
def pdf2md_single(source) :
    converter = DocumentConverter(format_options={ InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
            )
        }
    )
    result = converter.convert(source)
    return result.document.export_to_markdown()


#file = open('pdf2md.md','w',encoding='utf-8')
#file.write(pdf2md_single("../unique_sxolika_pdf/paper_16.pdf"))

In [3]:
def paragraph_maker(text,maxpadding = 2) :
    lines = text.splitlines()
    paragraphs = []
    emptyline = 0
    paragraph = ''
    for i,line in enumerate(lines) :
        if i == len(lines) - 1 :
            paragraph = paragraph + line
            paragraphs.append(paragraph)
            continue
        if line == '' :
            emptyline = emptyline + 1
            if emptyline == maxpadding :
                if paragraph != '' :
                    paragraphs.append(paragraph)
                emptyline = 0
                paragraph = ''
        else :
            paragraph = paragraph + line + '\n'
            emptyline = 0
            if i == len(lines) - 1 :
                paragraphs.append(paragraph)
    return paragraphs

def paragraph_merger(paragraphs,min_par_size,threshold) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if len(paragraph) < min_par_size :
            if paragraph != paragraphs[-1] :
                if len(paragraphs[i+1]) > threshold :
                    if not (paragraphs[i+1].startswith('##') or paragraphs[i+1].startswith('|') ) :
                        paragraphs[i+1] = paragraph + paragraphs[i+1]
                        continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_image(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('<!-- image -->') :
            continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_dotlines(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('.....') :
            if paragraph.endswith('.....') :
                continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_remove_artifacts(paragraphs,min_length = 5) :
    newparagraphs = []
    for paragraph in paragraphs :
        if len(paragraph) <= min_length :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_not_char_end(paragraph,chars,print) :
    #if not paragraph.startswith('##') :
    end_search_flag = False
    if not paragraph.endswith(chars) :
        if print :
            paragraph = '[Out:Paragraph does not end with specified char]' + paragraph
        else : paragraph = ''
        end_search_flag = True
    return (paragraph,end_search_flag)

def all_paragraph_not_char_end(paragraphs,chars,print = False) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        newparagraphs.append(paragraph_not_char_end(paragraph,chars,print)[0])
        if paragraph_not_char_end(paragraph,chars,print)[1] == False :
            for t_paragraph in paragraphs[i+1:] :
                newparagraphs.append(t_paragraph)
            return newparagraphs
    return newparagraphs

def remove_content_table_begin(paragraphs,num_of_front = 20,print = False) :
    newparagraphs = []
    table_starting_char = ('|','|')
    for paragraph in paragraphs[:num_of_front] :
        if paragraph.startswith(table_starting_char) :
            if print :
                paragraph = '[Out:Conent Table]' + paragraph
                newparagraphs.append(paragraph)
        else :
            newparagraphs.append(paragraph)
    for paragraph in paragraphs[10:] :
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_fix_broken_line(paragraphs) :
    ending_lower_pattern = re.compile(r'(.*[α-ωά-ώa-zΑ-ΩΆ-Ώ]\n)|(.*[0-9]\n)|(.*°C\n)|(.*\)\n)|(.*\-\n)|(.*,\n)')
    starting_lower_pattern = re.compile(r'([α-ωά-ώa-z])|([0-9])|(°)|\)|- ')
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if re.match(ending_lower_pattern,paragraph) :
            if paragraph != paragraphs[-1] :
                if re.match(starting_lower_pattern,paragraphs[i+1]) :
                    paragraphs[i+1] = paragraph + paragraphs[i+1]
                    continue
        newparagraphs.append(paragraph)
    return newparagraphs

def print_text(paragraphs) :
    for paragraph in paragraphs :
        print(paragraph)
        print('\n\n\n')

def remove_ending_chunk(paragraphs) :
    paragraphs = paragraphs[::-1]
    for i,paragraph in enumerate(paragraphs) :
        if paragraph.startswith('##') :
            paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
            return paragraphs[::-1]
        paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
        #del paragraphs[-1]
    return paragraphs[::-1]

def remove_taged_paragraphs(paragraphs,tags,print = False) :
    newparagraphs = []
    bibliography_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(tags) :
            bibliography_flag = True
            if print :
                paragraph = '[Out:Paragraph is' + tags[0] + ']' + paragraph
            continue
        elif bibliography_flag == True :
            if paragraph.startswith('##') and not paragraph.startswith(tags) :
                bibliography_flag = False
                newparagraphs.append(paragraph)
                continue
            if print :
                paragraph = '[Out:Paragraph is' + tags[0] + ']' + paragraph
            else : continue
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_noise(paragraphs,pattern) :
    newparagraphs = []
    for paragraph in paragraphs :
        paragraph = re.sub(pattern,'',paragraph)
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_contained_pattern(paragraphs,pattern,print = False) :
    newparagraphs = []
    for paragraph in paragraphs :
        if re.search(pattern,paragraph):
            if print :
                paragraph = '[Out: Pattern found]' + paragraph
            else : continue
        newparagraphs.append(paragraph)
    return newparagraphs

def test_write_text(paragraphs,file) :
    text = '[Paragraph Begin: Number 0 ]'
    for i,paragraph in enumerate(paragraphs) :
        text = text + paragraph + '[Paragraph End]\n\n\n[Paragraph Begin: Number ' + str(i+1) + ' ]'
    file.write(text)
    
def write_text(paragraphs,file) :
    text=''
    for i,paragraph in enumerate(paragraphs) :
        text = text + paragraph + '\n'
    file.write(text) 

In [4]:
endings = ('.\n','?\n','!\n',';\n','.\n',';\n','|\n')
bibliography_tags = ('## ΞΕΝΟΓΛΩΣΣΗ','## ΕΛΛΗΝΙΚΗ','## Bl ΒΛΙΟΓΡΑΦΙΑ', '## ΒΛΙΟΓΡΑΦΙΑ','## Π Ι ΝΑ Κ Α Σ Π Ρ Ο Ε Λ Ε Υ Σ Η Σ Τ Ω Ν E Ι ΚΟ Ν Ω Ν',
                         '## ΠΙΝΑΚΑΣ ΠΡΟΕΛΕΥΣΗΣ ΤΩΝ EΙΚΟΝΩΝ','## ΒΙΒΛΙΟΓΡΑΦΙΑ','## ΕΙΚΟΝΟΓΡΑΦΙΚΟ ΥΛΙΚΟ','## Διαδικτυακοί τόποι','## ΚΕΙΜΕΝΑ',
                         '## Ηλεκτρονικές διευθύνσεις στο Internet','## Ξενόγλωσση','## Ελληνική','## ΠΑΡΑΡΤΗΜΑ','## Στ. Φωτογραφικά άλμπουμ',
                         '## Θ. Ιστότοποι','## www.','## Η. Λοιπές μελέτες','## Επιλογή Βιβλιογραφίας','## ΠΑΙΔΑΓΩΓΙΚΕΣ ΜΕΛΕΤΕΣ','## ΠΗΓΕΣ ΦΩΤΟΓ ΡΑΦΙΚΟΥ ΥΛΙΚΟΥ',
                         '## Βιβλιογραφία','## 7. Εικονική βιβλιοθήκη Επιφανειοδραστικών','## ΞΕΝΗ','## Βιβλιογραφικά Βοηθήματα','## ΠΑΡΑΠΟΜΠΕΣ',
                         '## 7. Εικονική βιβλιοθήκη Επιφανειοδραστικών','## ΠΗΓΕΣ ΦΩΤΟΓΡΑΦΙΩΝ','## ΠΗrΕΣ προέλευσης','## Πηγές βοηθητικού υλικού','## Θεωρία',
                         '## Εικονογραφικό υλικό','## Ευχαριστούμε τους παρακάτω εκδοτικούς οίκους','## ΠPOEΛEYΣH TOY EIKONOΓPAΦIKOY YΛIKOY','## Πηγές εικόνων',
                         '## Πρόσθετο εικονιστικό υλικό','ΠΗΓΕΣ ΕΠΟΠΤΙΚΟΥ ΥΛΙΚΟΥ','## ΠΗΓΕΣ ΥΛΙΚΟΥ','Πίνακας συνδέσμων','## Βιβλία-Περιοδικά Ελληνόγλωσσα',
                         '## Ξενόγλωσσα','## ΣΗΜΕΙΩΣΕΙΣ ','## Κλασικά','## Φωτογραφίες και πίνακες που χρησιµοποιήθηκαν','## Ενδεικτική Βιβλιογραφία',
                         '## Ενδεικτική βιβλιογραφία','## βιβλιογραφία','## ΠΗΓΕΣ ΕΠΟΠΤΙΚΟΥ ΥΛΙΚΟΥ','## Εικονιστικό Υλικό','## Πίνακας συνδέσμων διαδικτυακών διευθύνσεων',
                         '## Βιβλία αναφοράς','## Σύνδεσμοι','## ΠΗΓΕΣ','## ΑΠΟ ΣΧΟΛΙΚΑ ΕΓΧΕΙΡΙΔΙΑ','## ΧΕΙΡΟΓΡΑΦΑ','## ΜΟΥΣΙΚΕΣ - ΜΟΥΣΙΚΟΛΟΓΙΚΕΣ ΜΕΛΕΤΕΣ','## Ξενόγλωσσες',
                         'Πηγή: ','## ΞΕΝΟΓΛΩΣΣΑ ΑΡΘΡΑ','## ΕΠΙΛΕΓΜΕΝΕΣ ΠΗΓΕΣ','## 13 Βιβλιογραφία - πηγές','## ΧΡΗΣΙΜΕΣ ΔΙΕΥΘΥΝΣΕΙΣ ΔΙΑΔΙΚΤΥΟΥ','## 8.6 Αναφορές - Βιβλιογραφία',
                         '## 8. Βιβλιογραφία','## ΠΗΓΕΣ ΕΙΚΟΝΩΝ','ΠΗΓΕΣ ΕΙΚΟΝΩΝ','## Ξένη','## Α ΒΙΒΛΙΑ','## ΜΕΤΑΦΡΑΣΜΕΝΗ','## ΞΕΝΗ','## Διεθνής Βιβλιογραφία',
                         '## ΔΙΚΤΥΑΚΟΙ ΤΟΠΟΙ','## ΒΙΒΛΙΟΓΡΑΦΙΚΕΣ ΠΗΓΕΣ','## Πηγές','## Bιβλιογραφία','## Εικόνες εσωφύλλου','## 7. Περιοδικά','## 6. Μη Κυβερνητικές Οργανώσεις',
                         '## 5. Ευρωπαϊκή Ένωση','## 4. Ειδικές Ιστοσελίδες','## 3. Διεθνείς Θεσµοί','## 2. Δηµόσιες και Ανεξάρτητες Υπηρεσίες, Αρχές και πολιτικά κόµµατα',
                         '## 1. Γενικοί Αναζητητές','## Ελληνόγλωσση','## α. Ελληνόγλωσση','## β. Ξενόγλωσση','## Ι. Ελληνική','## II. Ξενόγλωσση','## 1.7. Βιβλιογραφία-Δικτυογραφία',
                         '## 2.3. Βιβλιογραφία - Δικτυογραφία','## 3.6. Βιβλιογραφία - Δικτυογραφία','## 4.7. Βιβλιογραφία - Δικτυογραφία','## 5.10. Βιβλιογραφία-Δικτυογραφία',
                         '## ΧΡΗΣΙΜΕΣ ΔΙΕΥΘΥΝΣΕΙΣ','## ΚΑΤΑΛΟΓΟΣ ΑΡΜΟΔΙΩΝ','## Τεχνικά φυλλάδια','## Περιοδικά και ενημερωτικά φυλλάδια','## Στοιχεία και πληροφορίες','## ΠΑΡΟΡΑΜΑΤΑ',
                         '## 5 - ΠΙΝΑΚΑΣ ΕΤΑΙΡΙΩΝ','## 4 - ΔΙΕΥΘΥΝΣΕΙΣ INTERNET','## 3 - ΤΕΧΝΙΚΑ ΠΕΡΙΟΔΙΚΑ','## 2 - ΠΡΟΤΥΠΑ - ΤΕΧΝΙΚΕΣ ΟΔΗΓΙΕΣ','## 1 - ΤΕΧΝΙΚΑ ΒΙΒΛΙΑ & ΕΓΧΕΙΡΙΔΙΑ',
                         '## Χρήσιμοι Διαδικτυακοί Τόποι','## Χαρακτηριστικές NoSQL Βάσεις Δεδομένων','## https:',
                         '## %CE%')
glossary_tags = ('## ΓΛΩΣΣΑΡΙ, ## Γλωσσάρι','## Λ EΞIKO','## ΛEΞIKO','## ΛΕΞΙΛΟΓΙΟ','## ΓΛΩΣΣΑ ΡΙ','## Ζ. Λε ξ ικά','## Ορολογία','## Γλωσσάρι','## ΓΛΩΣΣΑΡΙΟ',
                     '## Συγκεντρωτικός πίνακας Γραµµατικής','## Β. Γλωσσάρι','## ΓΛΩΣΣΑΡΙ','## ΛΕΞΙΚΟ',']## ορολογίας στην Ελληνική και την Αγγλική','## Λεξικό',
                     '-Λ ΕΞΙΛΟΓΙΟ ΟΡΩΝ','##  Γλωσσάρι')
euritirio_tags = ('## EYPETHPIA','## ΕΥΡΕΤΗΡΙΟ','## Ευρετήριο','## II. Eυρετήριο όρων','## Αλφαβητικό ευρετήριο όρων','## ευρετήριο','## EYPETHPIO')
church_tags = ('## Δ. Λειτουργικά Β ι β λία','## Β . Μουσικά β ι β λία','## Ε. Πατερικά Έργα','## e-kere.gr','## Ι. Π η γές Εικόνων',
                   '## Α\' Γ Υ Μ Ν Α Σ Ι Ο Υ','## Α. Θεωρ η τικά Εκκλ η σιαστικ ή ς Μουσικ ή ς','## Ένα ταξίδι ζωής: Η συνάντηση Θεού και ανθρώπου μέσα από τις βιβλικές διηγήσεις',
                   '## Η Εκκλησία: πορεία ζωής μέσα στην ιστορία','## Η μαρτυρία της Ορθόδοξης Εκκλησίας στον σύγχρονο κόσμο','Εικονογράφηση εξωφύλλου','## Χριστιανισμός και Θρησκεύματα',
                   '## Εισαγωγική σημείωση','## ΘΕΟΛΟΓΙΚΕΣ ΜΕΛΕΤΕΣ','## ΕΝΤΥΠΕΣ ΣΥΛΛΟΓΕΣ ΕΚΚΛΗΣΙΑΣΤΙΚΗΣ ΜΟΥΣΙΚΗΣ','## Κ ΑΤΑ ΛΟΓΟΙ','## Γ. Κατάλογοι μουσικών χειρογράφων',
                   '## χείο Σχολείου Ψαλτικής')
content_tags = ('## ΠΕΡΙΕΧΟΜΕΝΑ','## ΠΕΡΙΕΧΟΜΕΝΟ','## Περιεχόμενο','## Περιεχόμενα','## ΠΙΝΑΚΑΣ ΠΕΡΙΕΧΟΜΕΝΩΝ','## Πίνακας Περιεχομένων','## Π Ε Ρ Ι Ε Χ Ο Μ Ε Ν Α','## ΠEPIEXOMENA',
                    '## ΠΕΡΙΕΧΟΜΕΝA','## Περιεχόµενα','## Π Ι ΝΑ Κ Α Σ Π Ε Ρ Ι Ε ΧΟ Μ Ε Ν Ω Ν','## Συνοπτικός πίνακας περιεχοµένου','Πίνακας Περιεχομένων',' Π Ε Ρ Ι Ε Χ Ο Μ Ε Ν Α   | Π Ε Ρ Ι Ε Χ Ο Μ Ε Ν Α')
printing_tags = ('## ΣΤΟΙΧΕΙΑ ΕΠΑΝΕΚ∆ΟΣΗΣ','## ΣΤΟΙΧΕΙΑ ΑΡΧΙΚΗΣ ΕΚ∆ΟΣΗΣ','## ΥΠΟΥΡΓΕΙΟ ΠΑΙΔΕΙΑΣ',
                     '## ΣΥΓΓΡΑΦΕΙΣ','## ΚΡΙΤΕΣ','## ΣΥΝΤΟΝΙΣΤΗΣ','## ΙΝΣΤΙΤΟΥΤΟ ΕΚΠΑΙΔΕΥΤΙΚΗΣ ΠΟΛΙΤΙΚΗΣ','## ΓΛΩΣΣΙΚΟΣ ΕΛΕΓΧΟΣ', '## Σχετικές Ηλεκτρονικές διευθύνσεις', 
                     '## Ηλεκτρονική επεξεργασία εικόνας' , '## Ψηφιακή σχεδίαση','## Ηλεκτρονική Τυπογραφία','## Ηλεκτρονική Τυπογραφία','## ΕΠΙΤΡΟΠΗ ΚΡΙΣΗΣ','## ΓΛΩΣΣΙΚΗ ΕΠΙΜΕΛΕΙΑ',
                     '## Επιτροπή αξιολόγησης','## Ομάδα αναθεώρησης','## Εποπτεία και συντονισμός αναθεώρησης','## Καλλιτεχνική επιμέλεια','## ΕΠΟΠΤΕΙΑ ΤΗΣ ΑΝΑΜΟΡΦΩΣΗΣ ΣΤΟ ΠΛΑΙΣΙΟ ΤΟΥ Π.Ι.',
                     '## ΕΠΙΜΕΛΕΙΑ','## Κωδικός Βιβλίου','## Υπεύθυνη Δράσης','## Υπεύθυνος για τo Παιδαγωγικό','## Φιλολογική επιμέλεια','## ΣΗΜΕΙΩΣΕΙΣ',
                     'ΙΝΣΤΙΤΟΥΤΟ ΕΚΠΑΙΔΕΥΤΙΚΗΣ ΠΟΛΙΤΙΚΗΣ','©','## ΣΥΝΤΟΝΙΣΤΡΙΑ','## ΗΛΕΚΤΡΟΝΙΚΗ ΕΠΕΞΕΡΓΑΣΙΑ','## Συντονιστική Επιτροπή του Έργου','## ΥΠΕΥΘΥΝΟΣ ΣΤΟ ΠΛΑΙΣΙΟ',
                     'Email:','## Συγγραφική ομάδα','## ΓΕΝΙΚΗ ΕΠΙΜΕΛΕΙΑ','ΣΤΟΙΧΕΙΑ ΕΠΑΝΕΚ∆ΟΣΗΣ','## Ηλεκτρονικές Διευθύνσεις','## ΠΑΙΔΑΓΩΓΙΚΟ ΙΝΣΤΙΤΟΥΤΟ','## Ομάδα συγγραφής',
                     '## Ομάδα κρίσης','## Γλωσσική επιμέλεια','## Συντονιστής','## ΕΥΧΑΡΙΣΤΙΕΣ','Email:','## Ηλεκτρονική επεξεργασία','## Συντονισμός','## Συγγραφική Ομάδα',
                     '## Στην παρούσα έκδοση','## Καλλιτεχνική Επιμέλεια','## Ανώτατα Εκπαιδευτικά Ιδρύματα','## Τεχνολογικά Εκπαιδευτικά Ιδρύματα','## Ευχαριστίες',
                     '## ΣΤΟΙΧΕΙΑ ΑΡΧΙΚΗΣ ΕΚΔΟΣΗΣ','## ΥΠΕΥΘΥΝΟΙ ΤΟΥ ΜΑΘΗΜΑΤΟΣ','## πραγματοποιήθηκε ΣΤΟΙΧΕΙΑ ΕΠΑΝΕΚ∆ΟΣΗΣ')
exception_tags = ('## ΜΙΚΡΟΒΙΟΛΟΓΙΑ','## <non-compliant-utf8-text>','## Εφαρμογές με Λογισμικό','## Συντομογραφίες','## ● ','## ΝΟΜΟΘΕΣΙΑ','## Ἔα δὴ','## Ενέργεια 2.3.2.',
                      '## Σεκλιζιώτης Σταμάτης','## Συντμήσεις','Ασημάκη Π., Θεοδωροπούλου Μ., Κωνσταντινίδου Ε., Μπόντια Χ.')
diofantos_pattern = re.compile(r'IΤΥΕ - ΔΙΟΦΑΝΤΟΣ|«ΔΙΟΦΑΝΤΟΣ|«Διόφαντος»|«ΔΙΟΦΑΝΤΟΣ»')
writer_pattern = re.compile(r'ΣΥΓΓΡΑΦΕΑΣ|Το παρόν εκπονήθηκε αμισθί και εγκρίθηκε|ΚΡΙΤΕΣ-ΑΞΙΟΛΟΓΗΤΕΣ|Σύμβουλος Παιδαγωγικού Ινστιτούτου.|Πρόεδρος:|ΣΥΝΤΟΝΙΣΜΟΣ:|Η συγγραφή και η επιστηµονική επιµέλεια του βιβλίου|Παροράματα βιβλίων Μηχανολογικού Τομέα')
religious_schoolbook_pattern = re.compile(r'Α\) Την εισήγηση για την πλήρη συμμόρφωση των Προγραμμάτων Σπουδών')
legal_note_3966_2011_pattern = re.compile(r'ν. 3966/2011')
noise_pattern = re.compile(r'$(.*?)$|\\_(\\_)+|Ú|È|Ì||Ô|È|Ù|Ë|Á|Ï|Ò|Û|·|Ó|Â|Î|‹|ˆ|∂ ¡ √ Δ ∏ Δ ∞|<non-compliant-utf8-text>')
funding_pattern = re.compile(r'συγχρηματοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση|συγχρηµατοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση|Έργο συγχρηματοδοτούμενο |Έργο συγχρηµατοδοτούµενο')
epal_pattern = re.compile(r'[ΑΓΒAB](\s)*\' ΕΠΑ\.Λ\.|B\'ΕΠΑ\.Λ\.|## Γ\'ΕΠΑ\.Λ\.')
gymnasium_pattern = re.compile(r'[ΑΒΓAB](\s)*\' Γυμνασίου')
exofilo_pattern = re.compile(r'Οι 3 εικόνες με φωτοαπόδοση που χρησιμοποιήθηκαν στο εξώφυλλο|Επεξήγηση του εξωφύλλου:|Στο κέντρο: Μαρσέλ|Αφοί ΤΖΙΦΑ Α.Ε.Β.Ε.')

In [5]:
os.makedirs('../test_output', exist_ok=True)
for i,file in enumerate(os.listdir('../md_output')) :
    if not file.endswith('.md'):
            continue
    try:
        with open('../md_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    paragraphs = paragraph_maker(text,maxpadding=1)
    paragraphs = paragraph_clean_image(paragraphs)
    paragraphs = paragraph_clean_dotlines(paragraphs)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    paragraphs = paragraph_fix_broken_line(paragraphs)
    paragraphs = paragraph_merger(paragraphs,500,10)
    paragraphs = remove_contained_pattern(paragraphs,diofantos_pattern)
    paragraphs = remove_contained_pattern(paragraphs,funding_pattern)
    paragraphs = remove_contained_pattern(paragraphs,legal_note_3966_2011_pattern)
    paragraphs = remove_contained_pattern(paragraphs,religious_schoolbook_pattern)
    paragraphs = remove_contained_pattern(paragraphs,writer_pattern)
    paragraphs = remove_contained_pattern(paragraphs,epal_pattern)
    paragraphs = remove_contained_pattern(paragraphs,gymnasium_pattern)
    paragraphs = remove_contained_pattern(paragraphs,exofilo_pattern)
    paragraphs = remove_taged_paragraphs(paragraphs,bibliography_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,euritirio_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,glossary_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,church_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,content_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,printing_tags)
    paragraphs =remove_taged_paragraphs(paragraphs,exception_tags)
    paragraphs = all_paragraph_not_char_end(paragraphs,endings)
    paragraphs = remove_content_table_begin(paragraphs,100)
    paragraphs = remove_content_table_begin(paragraphs[::-1])[::-1]
    paragraphs = remove_noise(paragraphs,noise_pattern)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    #paragraphs = remove_ending_chunk(paragraphs)
    
    t_file = 'test_'+file

    output_file_path = os.path.join('../test_output', t_file)
    try:
        with open(output_file_path, 'w', encoding='utf-8') as output_file:
            write_text(paragraphs,output_file)
    except Exception as e:
            print(f"Error writing to {output_file_path}: {e}")
            continue

In [6]:
import pandas as pd

textlist = []

for i,file in enumerate(os.listdir('../test_output')) :
    if not file.endswith('.md'):
            continue
    try:
        with open('../test_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue
    textlist.append(text)
    
df = pd.DataFrame({'Text':textlist})
print(df)
df.to_csv('Schoolbook_clean.csv')
df.to_parquet(path='Schoolbook_clean.parquet')

                                                  Text
0    ## Α$\_{'}$ ΤΕΥΧOΣ  Πετώντας µε τις λέξεις\n\n...
1    ## Κείµενα:\n- 1. Γράµµα στον Τσάρλι Τσάπλιν (...
2    ## κάποιος είναι δειλός, όλα τα κάνει με δειλί...
3    ## Πρόλογος\nΣτόχος του εγχειριδίου είναι να ο...
4    Β. Κατσαργύρης, Σ. Παπασταυρίδης, Γ. Πολύζος κ...
..                                                 ...
118  ## Σκεπτικό - σύντομη περιγραφή\n- Ο θεματικός...
119  ## Πρόλογος\nΤο βιβλίο αυτό περιλαμβάνει την ι...
120  ## EΙΣΑΓΩΓΗ\nΤο ανεξάρτητ ο νεοελληνικό κράτος...
121  ## ΞΕΚΙΝΩΝΤΑΣ ΑΠΟ ΤΗΝ ΑΠΟΡΙΑ\nΤι είναι φιλοσοφ...
122  ## Δύο λόγια για το βιβλίο\nΤο βιβλίο που κρατ...

[123 rows x 1 columns]
